## 1.Loading Needed Libraries

In [1]:
import joblib
import implicit
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from implicit.nearest_neighbours import bm25_weight
from implicit.evaluation import precision_at_k, ndcg_at_k, AUC_at_k, mean_average_precision_at_k

## 2.Loading the Data

In [2]:
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')

#### Creating Dev Set

In [3]:
val_cutoff = test['timestamp'].quantile(0.5)

val = test[test['timestamp'] < val_cutoff]
test = test[test['timestamp'] >= val_cutoff]

## 3.Creating the Matrix

In [4]:
num_items = pd.concat([train['item_idx'], test['item_idx']]).max() + 1
num_users = pd.concat([train['user_idx'], test['user_idx']]).max() + 1

train_item_user_matrix = sparse.csr_matrix(
    (train['weight'], (train['item_idx'], train['user_idx'])),
    shape=(num_items, num_users)
)

val_item_user_matrix = sparse.csr_matrix(
    (val['weight'], (val['item_idx'], val['user_idx'])),
    shape=(num_items, num_users)
)

test_item_user_matrix = sparse.csr_matrix(
    (test['weight'], (test['item_idx'], test['user_idx'])),
    shape=(num_items, num_users)
)

train_user_item_matrix = train_item_user_matrix.T.tocsr()
val_user_item_matrix = val_item_user_matrix.T.tocsr()
test_user_item_matrix = test_item_user_matrix.T.tocsr()

## 4.Checking best score model

In [ ]:
best_score = 0
best_params = None

for fac in [64, 128, 256]:
    for reg in [0.5, 1, 5, 10]:
        for iteration in [15, 25]:
            model = implicit.als.AlternatingLeastSquares(factors=fac, regularization=reg, iterations=iteration)
            model.fit(train_user_item_matrix)
            score_overall = ndcg_at_k(model, train_user_item_matrix, val_user_item_matrix, K=10)
            if score_overall > best_score:
                best_score = score_overall
                best_params = (fac, reg, iteration)


## 5.Training model on best parameters

In [5]:
train_bm25 = bm25_weight(train_user_item_matrix, K1=100, B=0.8).tocsr()

model = implicit.als.AlternatingLeastSquares(factors=256, regularization=0.5, iterations=15, random_state=42)
model.fit(train_bm25)

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

## 5. Saving the Model

In [6]:
joblib.dump(model, '../models/als.pkl')
print('Saved Succesfully!')

Saved Succesfully!


## 6.Evaluation

In [7]:
print(f'Best paramteres have gotten from the run:')
print(f'Best factor: 256')
print(f'Best regularization: 0.5')
print(f'Best iteration: 15')

Best paramteres have gotten from the run:
Best factor: 256
Best regularization: 0.5
Best iteration: 15


In [8]:
precision = precision_at_k(model, train_bm25, test_user_item_matrix, K=10)
ndcg = ndcg_at_k(model, train_bm25, test_user_item_matrix, K=10)
map_score = mean_average_precision_at_k(model, train_bm25, test_user_item_matrix, K=10)
auc = AUC_at_k(model, train_bm25, test_user_item_matrix, K=10)

print(f"Precision@10: {precision:.4f}")
print(f"NDCG@10: {ndcg:.4f}")
print(f"MAP@10: {map_score:.4f}")
print(f"AUC@10: {auc:.4f}")

  0%|          | 0/4690 [00:00<?, ?it/s]

  0%|          | 0/4690 [00:00<?, ?it/s]

  0%|          | 0/4690 [00:00<?, ?it/s]

  0%|          | 0/4690 [00:00<?, ?it/s]

Precision@10: 0.0323
NDCG@10: 0.0207
MAP@10: 0.0151
AUC@10: 0.5163
